# Research QuantBook: Options Wheel Tech Stocks## ObjectifAnalyser la strategie Wheel sur les actions tech disponibles en Docker.## Strategie- **Underlyings**: AAPL, MSFT, GOOGL (Tech stocks disponibles en Docker)- **Phase 1 (Cash)**: Vendre des puts OTM (~5-11% OTM)- **Phase 2 (Invested)**: Si assignation, vendre des calls couverts- **DTE**: 30 jours- **Max exposition**: 20% du portfolio par equity## Performance de referenceSharpe ~1.5-2.0 (2020-2025) - excellente sur les tech a haute volatilite.## Hypotheses a tester1. DTE: 21, 30, 45 jours2. OTM threshold: 3%, 5%, 7%3. Max exposition: 10%, 20%, 30%## Prerequis- Environnement Lean Research- Donnees actions + approximation option premium- Duree estimee: ~8 minutes## NoteLes options completes ne sont pas disponibles dans QuantBook. Ce notebook utiliseune approximation du premium basee sur la volatilite historique et le delta.Les tickers originaux (NVDA, ORCL, CSCO, AMD, QCOM) ne sont pas disponibles dansle Docker quantconnect/research:latest. Cette version utilise AAPL, MSFT, GOOGL.

In [1]:
# Setup QuantBook
from AlgorithmImports import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

qb = QuantBook()
print("QuantBook initialisé.")

QuantBook initialisé.


## 1. Chargement des donnees

On charge les donnees actions tech (AAPL, MSFT, GOOGL) disponibles en Docker pour la periode 2020-2026.

In [2]:
# Univers Tech Stocks disponibles en Docker# Docker quantconnect/research:latest fournit uniquement:# AAPL, MSFT, GOOGL (tech stocks)# NVDA, ORCL, CSCO, AMD, QCOM ne sont PAS disponibles.tickers = ["AAPL", "QQQ", "GOOGL"]otm_addons = {"AAPL": 0.05, "QQQ": 0.06, "GOOGL": 0.07}symbols = {}for ticker in tickers:    symbols[ticker] = qb.add_equity(ticker, Resolution.DAILY).symbol# Charger l historique (2020-2026 pour bull market + volatilite)start = datetime(2020, 1, 1)end = datetime(2026, 1, 1)history = qb.history(list(symbols.values()), start, end, Resolution.DAILY)print(f"Donnees chargees: {len(history)} lignes")

Pivot de la serie close en DataFrame large, avec remapping des colonnes Symbol ticker pour Options-VGT.

In [3]:
# Pivoter les données
closes = history['close'].unstack(level=0)

symbol_to_ticker = {str(v): k for k, v in symbols.items()}
closes.columns = [symbol_to_ticker.get(str(c), str(c)) for c in closes.columns]
closes = closes.dropna()

print(f"Période: {(closes.index[0][-1] if isinstance(closes.index[0], tuple) else closes.index[0]).date()} à {(closes.index[-1][-1] if isinstance(closes.index[-1], tuple) else closes.index[-1]).date()}")
print(f"Données: {len(closes)} jours de trading")
print(f"Actions: {list(closes.columns)}")
print(f"\nStatistiques des prix finaux:")
for ticker in tickers:
    if ticker in closes.columns:
        ret = (closes[ticker].iloc[-1] / closes[ticker].iloc[0] - 1) * 100
        vol = closes[ticker].pct_change().std() * np.sqrt(252)
        print(f"  {ticker}: {ret:+.1f}% (vol: {vol:.1%})")

NameError: name 'history' is not defined

## 2. Approximation du premium d'options

Comme les options complètes ne sont pas disponibles dans QuantBook, on utilise une approximation basée sur la volatilité historique et Black-Scholes.

In [4]:
def estimate_option_premium(stock_price, historical_vol, days_to_expiry, 
                            otm_pct=0.05, option_type='put'):
    """
    Estime le premium d'une option via Black-Scholes simplifié.
    
    Args:
        stock_price: Prix actuel de l'action
        historical_vol: Volatilité historique annualisée
        days_to_expiry: Jours jusqu'à expiration
        otm_pct: % OTM (0.05 = 5% OTM)
        option_type: 'put' ou 'call'
    """
    # Paramètres
    vol = historical_vol
    t = days_to_expiry / 365.0
    r = 0.05  # Taux sans risque
    
    # Strike OTM
    if option_type == 'put':
        strike = stock_price * (1 - otm_pct)
    else:
        strike = stock_price * (1 + otm_pct)
    
    # Black-Scholes
    d1 = (np.log(stock_price / strike) + (r + 0.5 * vol**2) * t) / (vol * np.sqrt(t))
    d2 = d1 - vol * np.sqrt(t)
    
    if option_type == 'put':
        premium = strike * np.exp(-r * t) * norm.cdf(-d2) - stock_price * norm.cdf(-d1)
    else:
        premium = stock_price * norm.cdf(d1) - strike * np.exp(-r * t) * norm.cdf(d2)
    
    return max(0, premium)

# Test de la fonction d'estimation
test_price = 150.0
test_vol = 0.35
test_dte = 30

put_premium = estimate_option_premium(test_price, test_vol, test_dte, 0.05, 'put')
call_premium = estimate_option_premium(test_price, test_vol, test_dte, 0.05, 'call')

print(f"Premium estimé pour action à ${test_price}, vol={test_vol:.1%}, DTE={test_dte}:")
print(f"  PUT (5% OTM): ${put_premium:.2f}")
print(f"  CALL (5% OTM): ${call_premium:.2f}")

Premium estimé pour action à $150.0, vol=35.0%, DTE=30:
  PUT (5% OTM): $2.67
  CALL (5% OTM): $3.31


### Interprétation: Premium d'options

- **Volatilité élevée** → Premium plus élevé (avantageux pour le vendeur)
- **OTM** → Plus OTM = premium moins élevé mais probabilité d'exercice plus faible
- **DTE** → Plus de temps = plus de valeur temps
- **Tech stocks** → Volatilité élevée = premiums attractifs

## 3. Backtest Wheel Strategy

Simulation de la stratégie avec:
- Vente de puts quand cash
- Vente de calls couverts quand investi
- DTE 30 jours
- Max exposition 20% par equity

In [5]:
def backtest_wheel_tech(closes, otm_addons, 
                       dte=30,
                       base_otm=0.05,
                       max_exposure=0.20,
                       position_fraction=0.20):
    """
    Backtest Wheel Strategy sur tech stocks.
    
    Retourne les métriques de performance.
    """
    portfolio_values = [1.0]
    cash = 1_000_000  # Starting cash
    
    # Position state
    shares_held = {ticker: 0 for ticker in closes.columns}
    option_positions = {ticker: None for ticker in closes.columns}  # (type, strike, premium_received, entry_day)
    
    # Calculate historical volatility for each stock
    lookback_vol = 60
    vols = {}
    for ticker in closes.columns:
        returns = closes[ticker].pct_change().dropna()
        vols[ticker] = returns.rolling(lookback_vol).std() * np.sqrt(252)
    
    warmup = lookback_vol + 10
    
    for i in range(warmup, len(closes)):
        current_prices = closes.iloc[i]
        prev_prices = closes.iloc[i-1]
        
        # Check options expiry (simplified: expire after DTE days)
        for ticker in closes.columns:
            if option_positions[ticker] is not None:
                opt_type, strike, premium_rec, entry_day = option_positions[ticker]
                days_held = i - entry_day
                
                if days_held >= dte:
                    # Option expired
                    current_price = current_prices[ticker]
                    
                    if opt_type == 'put':
                        if current_price < strike:
                            # Assigned: buy shares at strike
                            cost = strike * 100
                            shares_held[ticker] += 100
                            cash -= cost
                        else:
                            # Expired worthless: keep premium
                            pass
                    elif opt_type == 'call':
                        if current_price > strike:
                            # Assigned: sell shares at strike
                            shares_sold = min(shares_held[ticker], 100)
                            proceeds = strike * shares_sold
                            shares_held[ticker] -= shares_sold
                            cash += proceeds
                        else:
                            # Expired worthless: keep premium
                            pass
                    
                    option_positions[ticker] = None
        
        # Sell new options
        for ticker in closes.columns:
            if option_positions[ticker] is not None:
                continue  # Already have position
            
            current_price = current_prices[ticker]
            current_vol = vols[ticker].iloc[i]
            
            if pd.isna(current_vol) or current_vol < 0.1:
                continue
            
            total_otm = base_otm + otm_addons.get(ticker, 0)
            
            if shares_held[ticker] >= 100:
                # Sell covered calls
                premium = estimate_option_premium(current_price, current_vol, dte, total_otm, 'call')
                contracts = min(shares_held[ticker] // 100, 10)
                if contracts > 0:
                    premium_received = premium * 100 * contracts
                    cash += premium_received
                    option_positions[ticker] = ('call', current_price * (1 + total_otm), premium_received, i)
            else:
                # Sell puts (if cash available)
                premium = estimate_option_premium(current_price, current_vol, dte, total_otm, 'put')
                required_cash = current_price * 100 * 0.5  # Margin approx
                exposure_per_equity = cash * position_fraction
                contracts = max(1, int(exposure_per_equity / (current_price * 100)))
                
                if cash > required_cash * contracts:
                    premium_received = premium * 100 * contracts
                    cash += premium_received  # Receive premium
                    option_positions[ticker] = ('put', current_price * (1 - total_otm), premium_received, i)
        
        # Calculate portfolio value
        stock_value = sum(shares_held[t] * current_prices[t] for t in closes.columns)
        port_value = cash + stock_value
        portfolio_values.append(port_value)
    
    # Métriques
    returns = np.diff(portfolio_values) / np.array(portfolio_values[:-1])
    cum_returns = pd.Series(portfolio_values[1:], index=closes.index[warmup:])
    starting_value = portfolio_values[0]
    normalized_values = [v / starting_value for v in portfolio_values[1:]]
    
    total_ret = (portfolio_values[-1] / portfolio_values[0]) - 1
    years = len(returns) / 252
    cagr = (1 + total_ret) ** (1 / years) - 1 if years > 0 else 0
    vol = np.std(returns) * np.sqrt(252) if len(returns) > 1 else 0
    sharpe = (cagr - 0.03) / vol if vol > 0.001 else 0
    
    running_max = pd.Series(normalized_values).expanding().max()
    drawdown = (pd.Series(normalized_values) - running_max) / running_max
    max_dd = drawdown.min()
    
    return {
        'cum': pd.Series(normalized_values, index=closes.index[warmup:]),
        'sharpe': sharpe,
        'cagr': cagr,
        'max_dd': max_dd,
        'vol': vol,
        'final_value': normalized_values[-1] if normalized_values else 1
    }

result = backtest_wheel_tech(closes, otm_addons)

print(f"Performance Wheel Tech:")
print(f"  Sharpe: {result['sharpe']:.3f}")
print(f"  CAGR:   {result['cagr']:.1%}")
print(f"  Max DD: {result['max_dd']:.1%}")
print(f"  Vol:    {result['vol']:.1%}")

NameError: name 'closes' is not defined

## 4. Test du DTE

In [6]:
# Test différentes valeurs de DTE
dte_values = [21, 30, 45]

print(f"{'DTE':<10} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8}")
print("-" * 38)

dte_results = {}
for dte in dte_values:
    r = backtest_wheel_tech(closes, otm_addons, dte=dte)
    dte_results[f"{dte}d"] = r
    print(f"{dte}j{'':<7} {r['sharpe']:>8.3f} {r['cagr']:>7.1%} {r['max_dd']:>7.1%}")

best_dte = max(dte_results.items(), key=lambda x: x[1]['sharpe'])
print(f"\nMeilleur DTE: {best_dte[0]} (Sharpe={best_dte[1]['sharpe']:.3f})")

DTE          Sharpe     CAGR    MaxDD
--------------------------------------


NameError: name 'closes' is not defined

## 5. Test du seuil OTM

In [7]:
## 6. Comparaison avec le benchmark Tech B&H

## 6. Comparaison avec VGT (ETF Tech) B&H

In [8]:
# Pour comparaison, créer un ETF tech "synthetic" (equal-weight)
tech_etf = closes.mean(axis=1)
warmup = 70
vgt_values = tech_etf.iloc[warmup:] / tech_etf.iloc[warmup]

# Métriques VGT B&H
vgt_ret = vgt_values.pct_change().dropna()
vgt_cagr = (vgt_values.iloc[-1] ** (252/len(vgt_values))) - 1
vgt_vol = vgt_ret.std() * np.sqrt(252)
vgt_sharpe = (vgt_cagr - 0.03) / vgt_vol
vgt_dd = (vgt_values / vgt_values.cummax() - 1).min()

print("=== Comparaison vs Tech ETF B&H ===")
print(f"{'Stratégie':<20} {'CAGR':>10} {'Sharpe':>10} {'MaxDD':>10}")
print("-" * 53)
print(f"{'Wheel Tech':<20} {result['cagr']:>9.1%} {result['sharpe']:>10.3f} {result['max_dd']:>9.1%}")
print(f"{'Tech ETF B&H':<20} {vgt_cagr:>9.1%} {vgt_sharpe:>10.3f} {vgt_dd:>9.1%}")

NameError: name 'closes' is not defined

## 7. Visualisation des résultats

In [9]:
## 8. Conclusions et recommandations### Resume| Metrique | Meilleure config ||----------|-----------------|| DTE | (a remplir) || OTM | (a remplir) || Sharpe | (a remplir) || CAGR | (a remplir) |### VerdictSi Sharpe > 1.2: **Deployer avec les parametres optimaux**### Points forts Wheel Tech- **Volatilite elevee**: Tech stocks = premiums attractifs- **Revenu regulier**: Ventes d options genèrent du revenu mensuel- **Diversification**: 3 stocks reduit le risque specifique- **Adaptatif**: OTM par stock selon sa volatilite### Limitations- **Assignment risk**: Possibilite d etre assigne sur les puts- **Gestion complexe**: Positions d options a gerer- **Capital intensif**: Necessite du capital pour les margin requirements### Prochaines etapes1. Deployer sur QC cloud avec les parametres optimaux2. Tester avec les tickers originaux (NVDA, AMD, etc.) sur QC Cloud3. Optimiser le rolling (avant expiration vs a expiration)4. Ajouter un filtre de volatilite (VIX) pour eviter les periodes extremes

## 8. Conclusions et recommandations

### Résumé

| Métrique | Meilleure config |
|----------|-----------------|
| DTE | (à remplir) |
| OTM | (à remplir) |
| Sharpe | (à remplir) |
| CAGR | (à remplir) |

### Verdict

Si Sharpe > 1.2: **Déployer avec les paramètres optimaux**

### Points forts Wheel Tech

- **Volatilité élevée**: Tech stocks = premiums attractifs
- **Revenu régulier**: Ventes d'options génèrent du revenu mensuel
- **Diversification**: 5 stocks réduit le risque spécifique
- **Adaptatif**: OTM par stock selon sa volatilité

### Limitations

- **Assignment risk**: Possibilité d'être assigné sur les puts
- **Gestion complexe**: 5 stocks = 10 positions d'options à gérer
- **Capital intensif**: Nécessite du capital pour les margin requirements

### Prochaines étapes

1. Déployer sur QC cloud avec les paramètres optimaux
2. Tester avec des stocks moins volatils (plus defensifs)
3. Optimiser le rolling (avant expiration vs à expiration)
4. Ajouter un filtre de volatilité (VIX) pour éviter les périodes extrêmes